# Project 5 — MOJ Criminal Court Statistics · **Silver layer**

Bronze was an honest photocopy of the source. **Silver is where I make it usable.**
Nothing here invents numbers — I only *reshape* the raw extract into clean, tidy
tables I can actually analyse: friendly column names, messy combined labels pulled
apart into their own columns, a real date, an ordered age category, and — critically —
guards against the two traps I found while profiling (stacked geography levels that
would double-count, and `Annual`/`All` aggregate rows).

I work one table at a time. Above each transform I say exactly what I'm renaming or
adding, so the change is never a surprise. At the end I rebuild the government's
80,203 headline from *two different* Silver tables to prove I didn't break anything.

## Step 0 — Tools and where things live

`IN_DIR` = the Bronze Parquet (my input). `OUT_DIR` = `data/silver` (my output).

In [1]:
import re
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

IN_DIR  = PROJECT_DIR / "data" / "bronze" / "parquet"
OUT_DIR = PROJECT_DIR / "data" / "silver"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Bronze in :", IN_DIR)
print("Silver out:", OUT_DIR)
print("pandas", pd.__version__)

Bronze in : /Users/yusufismail/moj-crown-court/data/bronze/parquet
Silver out: /Users/yusufismail/moj-crown-court/data/silver
pandas 2.2.2


## Step 1 — Shared helper tools I'll reuse on every table

I define four small helpers here so I write the logic once:

- **`add_time(df)`** — adds `year` (a real number), `quarter`, and `quarter_end`
  (an actual date, e.g. Q4 2025 → `2025-12-31`, because the caseload is a snapshot
  at quarter end). Time-series charts need a real date, not the text "Q4".
- **`split_offence(label)`** — turns `"02: Sexual offences - adult rape"` into a clean
  `offence_code` (`"02"`) and a canonical `offence` (`"Sexual offences - Adult Rape"`),
  and unifies `"Unknown"`/`"Not known"` → `"Not known"`. This fixes the spelling
  differences between files so the offence column *joins*.
- **`geo_level(area)`** — labels an area as `national` / `country` / `region` / `lja`
  / `unknown`. This is my seatbelt against double-counting: I filter to ONE level, I
  never sum across levels.
- **`strip_code(label)`** — drops a leading `"04. "` / `"02e. "` numbering prefix.

One efficiency habit baked in: these label columns only have a *handful* of distinct
values (e.g. 18 offence labels), even across millions of rows. So I clean the **unique
values once** and map the result back — never run the parser 4 million times. That's
what `map_unique` / `map_pair` do.

In [3]:
QEND = {"Q1": "-03-31", "Q2": "-06-30", "Q3": "-09-30", "Q4": "-12-31"}

def map_unique(series, func):
    """Apply func to the distinct values only, then map back onto every row (fast)."""
    lut = {v: func(v) for v in series.dropna().unique()}
    return series.map(lut)

def map_pair(series, func):
    """Same idea, but func returns two things -> return two mapped columns."""
    u = series.dropna().unique()
    a = {v: func(v)[0] for v in u}
    b = {v: func(v)[1] for v in u}
    return series.map(a), series.map(b)

def add_time(df, ycol="year", qcol="quarter"):
    df = df.copy()
    df["year"] = df[ycol].astype(int)                       # "2025" -> 2025
    df["quarter"] = df[qcol]
    df["quarter_end"] = pd.to_datetime(df["year"].astype(str) + df["quarter"].map(QEND))
    return df

def strip_code(label):
    # remove a leading code like "04. " or "02e. " or "13: "
    return re.sub(r"^[0-9A-Za-z]+[.:]\s*", "", str(label)).strip()

def split_offence(label):
    label = str(label)
    code = label.split(":", 1)[0].strip() if ":" in label else None
    name = label.split(":", 1)[1].strip() if ":" in label else label
    if name.lower() in ("not known", "unknown"):           # unify the two spellings
        name = "Not known"
    if " - " in name:                                       # unify rape sub-type casing
        base, sub = name.split(" - ", 1)
        name = f"{base} - {sub.title()}"                    # "adult rape" -> "Adult Rape"
    return code, name

def geo_level(area):
    a = str(area)
    if a.endswith("LCJB"):        return "lja"
    if a == "England and Wales":  return "national"
    if a == "England":            return "country"
    if a in ("Unknown", "Unknown region"): return "unknown"
    return "region"               # London / Midlands / North East ... / Wales

print("Helpers ready: add_time, strip_code, split_offence, geo_level")

Helpers ready: add_time, strip_code, split_offence, geo_level


## Step 2 — Table 1: `silver_cc_rdos` (the backlog volumes)

This is the receipts / disposals / open-caseload file, and it's **leaf-level** — one
row per individual court. National figures come from summing up, so there are no
subtotal rows to strip.

**What I'm changing (and the new names):**
- `rdos` → **`measure`**, values cleaned to `Receipts` / `Disposals` / `Open`
- `receipt_type` → **`case_type`** (prefix stripped, e.g. `Indictable only trials`)
- `offence_group` → **`offence_code`** + **`offence`** (canonical)
- `lcjb_area` → **`lja`**, `crown_court` → **`court`** (blanks → `Unknown`)
- `value` → **`count`**
- plus `year`, `quarter`, `quarter_end` from `add_time`

In [5]:
rdos = pd.read_parquet(IN_DIR / "cc_rdos.parquet")

s = add_time(rdos)
s["measure"]   = s["rdos"].map({"1. Receipts": "Receipts",
                                "2. Disposals": "Disposals",
                                "3. Open": "Open"})
s["case_type"] = map_unique(s["receipt_type"], strip_code)
s["offence_code"], s["offence"] = map_pair(s["offence_group"], split_offence)
s = s.rename(columns={"lcjb_area": "lja", "crown_court": "court", "value": "count"})
s["court"] = s["court"].fillna("Unknown")

silver_rdos = s[["quarter_end", "year", "quarter",
                 "region", "lja", "court",
                 "case_type", "offence_code", "offence",
                 "measure", "count",
                 "_source_file", "_release"]].copy()

silver_rdos.to_parquet(OUT_DIR / "silver_cc_rdos.parquet", index=False)
print("silver_cc_rdos:", silver_rdos.shape)
silver_rdos.head(4)

silver_cc_rdos: (345590, 13)


,quarter_end,year,quarter,region,lja,court,case_type,offence_code,offence,measure,count,_source_file,_release
0,2016-03-31,2016,Q1,London,Central London LCJB,Central Criminal Court,Triable-either-way trials: total,00,All offences,Receipts,63.0,cc_rdos_tool.xlsx,2025Q4
1,2016-03-31,2016,Q1,London,Central London LCJB,Central Criminal Court,Triable-either-way trials: total,01,Violence against the person,Receipts,4.0,cc_rdos_tool.xlsx,2025Q4
2,2016-03-31,2016,Q1,London,Central London LCJB,Central Criminal Court,Triable-either-way trials: total,02,Sexual offences,Receipts,1.0,cc_rdos_tool.xlsx,2025Q4
3,2016-03-31,2016,Q1,London,Central London LCJB,Central Criminal Court,Triable-either-way trials: total,04,Theft offences,Receipts,4.0,cc_rdos_tool.xlsx,2025Q4


## Step 3 — Table 2: `silver_cc_open_age` (age of the open caseload)

This is the most important table for the story (the "waiting a year or more" angle),
and it has the trickiest structure, so I go carefully.

**What I'm changing (and the new names):**
- `geographic_area` → **`geo_area`** + a new **`geo_level`** tag
  (national / country / region / lja) — my double-count seatbelt.
- `receipt_type` → **`case_type`** + **`remand_status`**. Example:
  `"04. Indictable only trials: remanded in custody"` becomes
  `case_type = "Indictable only trials"`, `remand_status = "Custody"`.
- `age_open_grouped` → **`age_band`** (prefix stripped) + **`age_order`** (1–9 so it
  sorts correctly) + **`row_type`** flag: `age_band` for the 9 real bands vs `summary`
  for the `Total cases` / `Valid cases` rows. This lets me keep the 80,203 *total*
  AND the age breakdown in one table without ever accidentally summing them together.
- `offence_group` → **`offence_code`** + **`offence`**; `value` → **`count`**.

In [8]:
open_age = pd.read_parquet(IN_DIR / "cc_open_age.parquet")

def parse_open_receipt(rt):
    body = re.sub(r"^[0-9A-Za-z]+\.\s*", "", str(rt))     # drop "04. " / "02e. "
    if ":" in body:
        case, rem = [p.strip() for p in body.split(":", 1)]
    else:
        case, rem = body.strip(), "total"                  # e.g. "All open cases"
    rem_map = {"total": "All", "remanded in custody": "Custody",
               "remanded on bail": "Bail", "remand status unknown": "Unknown"}
    return case, rem_map.get(rem.lower(), rem)

AGE_ORDER = {
    "Under 4 weeks": 1, "4 to 8 weeks": 2, "8 to 12 weeks": 3, "12 to 16 weeks": 4,
    "16 to 20 weeks": 5, "20 to 26 weeks": 6, "6 months to under 1 year": 7,
    "1 to 2 years": 8, "2 years or more": 9,
}

o = add_time(open_age)
o["geo_area"]  = o["geographic_area"]
o["geo_level"] = map_unique(o["geo_area"], geo_level)
o["case_type"], o["remand_status"] = map_pair(o["receipt_type"], parse_open_receipt)
o["age_band"]  = map_unique(o["age_open_grouped"], strip_code)
o["age_order"] = o["age_band"].map(AGE_ORDER)              # NaN for Total/Valid rows
o["row_type"]  = map_unique(o["age_band"],
                            lambda a: "summary" if a in ("Total cases", "Valid cases") else "age_band")
o["offence_code"], o["offence"] = map_pair(o["offence_group"], split_offence)
o = o.rename(columns={"value": "count"})

# make age_band an ORDERED category so charts/sorts respect the real order
band_cats = sorted(AGE_ORDER, key=AGE_ORDER.get) + ["Total cases", "Valid cases"]
o["age_band"] = pd.Categorical(o["age_band"], categories=band_cats, ordered=True)

silver_open = o[["quarter_end", "year", "quarter",
                 "geo_area", "geo_level",
                 "case_type", "remand_status",
                 "offence_code", "offence",
                 "age_band", "age_order", "row_type", "count",
                 "_source_file", "_release"]].copy()

silver_open.to_parquet(OUT_DIR / "silver_cc_open_age.parquet", index=False)
print("silver_cc_open_age:", silver_open.shape)
print("geo_level counts:\n", silver_open.geo_level.value_counts())
silver_open.head(4)

silver_cc_open_age: (3129789, 15)
geo_level counts:
 geo_level
lja         2395322
region       541074
national      94928
country       94696
unknown        3769
Name: count, dtype: int64


,quarter_end,year,quarter,geo_area,geo_level,case_type,remand_status,offence_code,offence,age_band,age_order,row_type,count,_source_file,_release
0,2016-03-31,2016,Q1,Avon and Somerset LCJB,lja,All open cases,All,00,All offences,Under 4 weeks,1.0,age_band,184.0,cc_open_tool.xlsx,2025Q4
1,2016-03-31,2016,Q1,Avon and Somerset LCJB,lja,All open cases,All,00,All offences,4 to 8 weeks,2.0,age_band,173.0,cc_open_tool.xlsx,2025Q4
2,2016-03-31,2016,Q1,Avon and Somerset LCJB,lja,All open cases,All,00,All offences,8 to 12 weeks,3.0,age_band,104.0,cc_open_tool.xlsx,2025Q4
3,2016-03-31,2016,Q1,Avon and Somerset LCJB,lja,All open cases,All,00,All offences,12 to 16 weeks,4.0,age_band,93.0,cc_open_tool.xlsx,2025Q4


## Step 4 — Table 3: `silver_cc_waiting_hearing` (waiting & hearing times)

Two jobs here. First, **drop the aggregate rows** (`Annual` view and the `quarter =
"All"` rows) so I only keep clean quarterly data and never double-count. Second,
**split the overloaded `measure` column** into `stat` (Mean / Median / Total / Valid)
and `unit` (weeks / hours / cases / defendants / hearings) — because waiting time is
in *weeks* but hearing time is in *hours*, and they must never be mixed.

**New names:** `waiting_hearing_times` → **`metric`** (Waiting/Hearing), `measure` →
**`stat`** + **`unit`**, `receipt_type` → **`case_type`**, cleaned `remand_status` and
`plea`, `value` → **`value`**, plus `geo_level`.

In [10]:
wh = pd.read_parquet(IN_DIR / "cc_waiting_hearing.parquet")

before = len(wh)
wh = wh[(wh["annual_quarterly"] == "Quarterly") & (wh["quarter"] != "All")].copy()
print(f"dropped {before - len(wh):,} Annual/All rows; {len(wh):,} quarterly rows remain")

def split_measure(m):
    body = re.sub(r"^\d+\.\s*", "", str(m))              # "2. Median (weeks)" -> "Median (weeks)"
    stat = re.match(r"(Median|Mean|Total|Valid)", body)
    stat = stat.group(1) if stat else body
    if   "(weeks)" in body:  unit = "weeks"
    elif "(hours)" in body:  unit = "hours"
    elif "cases" in body:    unit = "cases"
    elif "defendants" in body: unit = "defendants"
    elif "hearings" in body: unit = "hearings"
    else:                    unit = "unknown"
    return stat, unit

w = add_time(wh)
w["metric"]        = map_unique(w["waiting_hearing_times"], strip_code)  # Waiting/Hearing times
w["stat"], w["unit"] = map_pair(w["measure"], split_measure)
w["case_type"]     = map_unique(w["receipt_type"], strip_code)
w["remand_status"] = map_unique(w["remand_status"], strip_code)
w["plea"]          = map_unique(w["plea"], strip_code)
w["geo_level"]     = map_unique(w["region"], geo_level)
w["offence_code"], w["offence"] = map_pair(w["offence_group"], split_offence)

silver_wh = w[["quarter_end", "year", "quarter",
               "region", "geo_level",
               "case_type", "remand_status", "plea",
               "offence_code", "offence",
               "metric", "stat", "unit", "value",
               "_source_file", "_release"]].copy()

silver_wh.to_parquet(OUT_DIR / "silver_cc_waiting_hearing.parquet", index=False)
print("silver_cc_waiting_hearing:", silver_wh.shape)
silver_wh.head(4)

dropped 965,095 Annual/All rows; 3,381,824 quarterly rows remain
silver_cc_waiting_hearing: (3381824, 16)


,quarter_end,year,quarter,region,geo_level,case_type,remand_status,plea,offence_code,offence,metric,stat,unit,value,_source_file,_release
95878,2016-03-31,2016,Q1,England,country,All cases closed,All remand statuses,All pleas,00,All offences,Hearing times,Mean,hours,3.374830,cc_waiting_hearing_tool.xlsx,2025Q4
95879,2016-03-31,2016,Q1,England,country,All cases closed,All remand statuses,All pleas,00,All offences,Waiting times,Total,defendants,33191.000000,cc_waiting_hearing_tool.xlsx,2025Q4
95880,2016-03-31,2016,Q1,England,country,All cases closed,All remand statuses,All pleas,00,All offences,Hearing times,Median,hours,0.866667,cc_waiting_hearing_tool.xlsx,2025Q4
95881,2016-03-31,2016,Q1,England,country,All cases closed,All remand statuses,All pleas,00,All offences,Waiting times,Mean,weeks,18.498648,cc_waiting_hearing_tool.xlsx,2025Q4


## Step 5 — Table 4: `silver_cc_timeliness` (end-to-end timeliness)

This file is **wide** — 24 stage measures across the columns (offence-to-charge,
charge-to-first-listing, … , offence-to-completion, each as a Mean and a Median). For
Silver I keep it wide (deciding whether to melt it into long form is a Gold-layer
choice), but I still clean it up.

**What I'm changing:** drop `Annual`/`All`; rename `Year`/`Quarter`/`Geographic area`/
`Receipt type`/`Offence group` to my house style; add `geo_level`; clean `case_type`;
canonical `offence`. The 24 measure columns keep their meaning but get tidy snake_case
names (e.g. `Offence to completion (Median)` → `offence_to_completion_median`).

In [14]:
tl = pd.read_parquet(IN_DIR / "cc_timeliness.parquet")

before = len(tl)
tl = tl[(tl["Annual or quarterly"] == "Quarterly") & (tl["Quarter"] != "All")].copy()
print(f"dropped {before - len(tl):,} Annual/All rows; {len(tl):,} quarterly rows remain")

tl = tl.rename(columns={"Year": "year", "Quarter": "quarter",
                        "Geographic area": "geo_area",
                        "Receipt type": "receipt_type",
                        "Offence group": "offence_group"})
t = add_time(tl)
t["geo_level"] = map_unique(t["geo_area"], geo_level)
t["case_type"] = map_unique(t["receipt_type"], strip_code)
t["offence_code"], t["offence"] = map_pair(t["offence_group"], split_offence)

# tidy the 24 wide measure columns into snake_case
def snake(c):
    c = c.replace("(Mean)", "mean").replace("(Median)", "median")
    c = re.sub(r"[^0-9A-Za-z]+", "_", c).strip("_").lower()
    return c
measure_cols = [c for c in t.columns if ("(Mean)" in c or "(Median)" in c
                or c.startswith("Number of"))]
rename_measures = {c: snake(c) for c in measure_cols}
t = t.rename(columns=rename_measures)

keep = (["quarter_end", "year", "quarter", "geo_area", "geo_level",
         "case_type", "offence_code", "offence"]
        + list(rename_measures.values())
        + ["_source_file", "_release"])
silver_tl = t[keep].copy()

silver_tl.to_parquet(OUT_DIR / "silver_cc_timeliness.parquet", index=False)
print("silver_cc_timeliness:", silver_tl.shape)
print("example measure columns:", list(rename_measures.values())[:6], "...")
silver_tl.head(3)

dropped 173,451 Annual/All rows; 581,344 quarterly rows remain
silver_cc_timeliness: (581344, 34)
example measure columns: ['number_of_defendants_whose_cases_have_completed', 'number_of_valid_defendants_whose_cases_have_completed', 'offence_to_charge_mean', 'offence_to_charge_median', 'charge_to_first_listing_mean', 'charge_to_first_listing_median'] ...


,quarter_end,year,quarter,geo_area,geo_level,case_type,offence_code,offence,number_of_defendants_whose_cases_have_completed,number_of_valid_defendants_whose_cases_have_completed,...,at_court_mean,at_court_median,charge_to_sending_to_the_crown_court_mean,charge_to_sending_to_the_crown_court_median,receipt_at_the_crown_court_to_completion_mean,receipt_at_the_crown_court_to_completion_median,charge_to_completion_at_the_crown_court_mean,charge_to_completion_at_the_crown_court_median,_source_file,_release
0,2016-03-31,2016,Q1,England and Wales,national,All cases closed,00,All offences,35221.0,26418.0,...,180.0,140.0,29.0,17.0,173.0,133.0,203.0,160.0,timeliness_tool_Crown_Court.xlsx,2025Q4
1,2016-06-30,2016,Q2,England and Wales,national,All cases closed,00,All offences,34046.0,23833.0,...,188.0,149.0,30.0,18.0,181.0,143.0,212.0,168.0,timeliness_tool_Crown_Court.xlsx,2025Q4
2,2016-09-30,2016,Q3,England and Wales,national,All cases closed,00,All offences,32262.0,21958.0,...,184.0,143.0,33.0,21.0,176.0,134.0,210.0,165.0,timeliness_tool_Crown_Court.xlsx,2025Q4


## Step 6 — Prove Silver didn't break the numbers

The whole point: reshaping must not change a single count. I rebuild the government's
**80,203** headline from **two different** Silver tables and check they agree:

1. From `silver_cc_rdos` — sum `Open` across every court (leaf-level → national).
2. From `silver_cc_open_age` — the `Total cases` summary row at national level.

If both say 80,203, the reshape is faithful. I also confirm the age bands sum to the
75,799 *valid* cases, and that no `Annual`/`All` rows slipped through.

In [16]:
# 1) national OPEN from the leaf-level rdos table
open_rdos = silver_rdos[(silver_rdos.year == 2025) & (silver_rdos.quarter == "Q4") &
                        (silver_rdos.measure == "Open") &
                        (silver_rdos.offence == "All offences")]["count"].sum()

# 2) national TOTAL from the age table (the summary row)
nat = silver_open[(silver_open.year == 2025) & (silver_open.quarter == "Q4") &
                  (silver_open.geo_level == "national") &
                  (silver_open.case_type == "All open cases") &
                  (silver_open.offence == "All offences")]
open_total = nat[nat.age_band == "Total cases"]["count"].sum()
valid_sum  = nat[nat.row_type == "age_band"]["count"].sum()
yr1        = nat[nat.age_band.isin(["1 to 2 years", "2 years or more"])]["count"].sum()

print(f"From silver_cc_rdos   (sum of Open over courts): {open_rdos:,.0f}")
print(f"From silver_cc_open   (Total cases, national)  : {open_total:,.0f}")
print(f"Sum of 9 age bands (= valid cases)             : {valid_sum:,.0f}   (expect 75,799)")
print(f"Open 1 year or more                            : {yr1:,.0f}")

assert int(open_rdos) == 80203, "rdos national Open != 80,203"
assert int(open_total) == 80203, "open_age Total cases != 80,203"
assert int(valid_sum) == 75799, "age bands != valid cases"

# no aggregate rows left behind
assert "All" not in silver_wh.quarter.unique()
assert "All" not in silver_tl.quarter.unique()
print("\n Both tables agree at 80,203, and no Annual/All rows remain — Silver is faithful. ")

From silver_cc_rdos   (sum of Open over courts): 80,203
From silver_cc_open   (Total cases, national)  : 80,203
Sum of 9 age bands (= valid cases)             : 75,799   (expect 75,799)
Open 1 year or more                            : 21,002

 Both tables agree at 80,203, and no Annual/All rows remain — Silver is faithful. 


## Step 7 — A quick double-count seatbelt check

Because `cc_open_age` stacks geography levels in one column, summing across levels is a
trap. Here's the seatbelt — and a subtle lesson it taught me: the sub-national rows
that *partition* the national total are the **regions plus an `Unknown` bucket** (cases
not assigned to a region). Region alone falls ~100 short; region **+ unknown** equals
the national figure exactly. So the rule is: filter to ONE `geo_level`, and don't
forget the `Unknown` bucket exists.

In [18]:
slice_ = silver_open[(silver_open.year == 2025) & (silver_open.quarter == "Q4") &
                     (silver_open.case_type == "All open cases") &
                     (silver_open.offence == "All offences") &
                     (silver_open.age_band == "Total cases")]

nat_val     = slice_[slice_.geo_level == "national"]["count"].sum()
region_only = slice_[slice_.geo_level == "region"]["count"].sum()
region_plus = slice_[slice_.geo_level.isin(["region", "unknown"])]["count"].sum()

print(f"national total         : {nat_val:,.0f}")
print(f"regions only           : {region_only:,.0f}   (falls short — an Unknown bucket exists)")
print(f"regions + Unknown      : {region_plus:,.0f}")
assert int(region_plus) == int(nat_val), "geography levels do not reconcile"
print("\nReconciles: national == regions + Unknown. Filter to ONE geo_level, never sum across.")

national total         : 80,203
regions only           : 80,102   (falls short — an Unknown bucket exists)
regions + Unknown      : 80,203

Reconciles: national == regions + Unknown. Filter to ONE geo_level, never sum across.


---
### Silver layer complete

I now have four analysis-ready tables in `data/silver/`, each with friendly names, a
real `quarter_end` date, a canonical `offence`, and the geography/aggregate traps
defused (`geo_level` tags, `Annual`/`All` rows removed, age bands ordered). I confirmed
the reshape is faithful by rebuilding the 80,203 outstanding-caseload figure from two
independent tables.

**Next — the Gold layer.** Here I'll cut small, purpose-built tables aimed at specific
stories: the national backlog trajectory (the road to 80,200), the ageing caseload and
its offence mix, timeliness drift, and the geography of the backlog — the exact cuts
that become the article series and the Streamlit app.